**Topic**: 
    
    Hybrid recommender system with Ollama gemma3:latest 7B text integration

**Dataset**: 

    2024 Movie len dataset
    N user = 4310
    N Movie = 44730 with genres and title metadata

**Method**: 
    
    1) Utilize Content-based and pyspark ALS hybrid weighted scores to generate item recommendations
    2) Feed metadata such as user/item latent factors, user past liked history, and weighted scores to Gemma 3:latest to get insights on:
        a) Explain specific user's movie genre preferences based on the latent factors
        b) How are the movie recommendations connected to the user's preferences
        c) How does the hybrid factor, alpha, affect the recommendation list
        d) Can the LLM generate recommendations better than the current hybrid system and why

**Limitations**:

    Due to limited RAM capacity in local machine, a reduced dataset is used in this study


In [1]:
#Loading data
import pandas as pd
import requests
import io
import zipfile


url = 'https://files.grouplens.org/datasets/movielens/ml_belief_2024_data_release_2.zip'

response = requests.get(url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
    #Print all files inside the zip:
    
#read the target files only:
    with zip_file.open("data_release/movies.csv") as f:
        df_movie = pd.read_csv(f)
    
    with zip_file.open('data_release/ratings_for_additional_users.csv') as f:
        additional_rating_df = pd.read_csv(f)

    with zip_file.open('data_release/user_rating_history.csv') as f:
        user_rating_df = pd.read_csv(f)
        
df_ratings = user_rating_df.drop('tstamp', axis=1)

#Due to overlimit of data, we will have to merge the movie dataset with the user rating dataset:
df_com = df_ratings.merge(df_movie, on='movieId', how='left')

print(f'Number of users in df_com: {df_com["userId"].nunique()}')
print(f'Number of movies in df_com: {df_com["movieId"].nunique()}')

#Check for any Null values in ratings:
print(f'Number of Na values in the rating column: {df_com["rating"].isna().sum()}')

#Let's remove those Na rows and some synethic ratings with negative values and re-count the number of users available:
df_com_cleaned = df_com[(df_com['rating'] >= 0) & (~df_com['rating'].isna()) & (~df_com['title'].isna())]

print(f'Number of Na values after cleaning inthe rating column: {df_com_cleaned["rating"].isna().sum()}')
print(f'Number of users after cleaning: {df_com_cleaned["userId"].nunique()}')
print(f'Number of movies after cleaning: {df_com_cleaned["movieId"].nunique()}')

#Create a reduced version of the dataset:
df_reduce = df_com_cleaned.sample(frac=0.5, random_state=42)
print(f'Number of users in reduced dataset: {df_reduce["userId"].nunique()}')
print(f'Number of movies in reduced dataset: {df_reduce["movieId"].nunique()}')

Number of users in df_com: 4418
Number of movies in df_com: 85170
Number of Na values in the rating column: 36521
Number of Na values after cleaning inthe rating column: 0
Number of users after cleaning: 4415
Number of movies after cleaning: 57079
Number of users in reduced dataset: 4310
Number of movies in reduced dataset: 44730
